# Notebook 01 — Whisper ASR Extraction

Extract token-level ASR confidence (`avg_logprob`) and transcripts from short
English audio clips placed in `../data/`.

**Inputs**: `data/*.wav` and `data/*.mp3`
**Output**: `output/01_whisper_results.csv` with columns
`file, transcript, avg_logprob, n_segments, duration_s`

**Hardware notes**:
- Loads `whisper-medium` (~1.5 GB on disk, ~2 GB VRAM) on CUDA if available.
- Restart this kernel before running notebook 02 to free VRAM for the LLM.


## 1. Imports and device selection

Pick CUDA when available; otherwise fall back to CPU (much slower, but still works).


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
import torch
import whisper

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## 2. Load Whisper model

`whisper-medium` is the default trade-off for our 6 GB VRAM target — it gives
substantially better English transcription than `small` while leaving headroom.
First call downloads the weights to `~/.cache/whisper/`.


In [ ]:
model = whisper.load_model("medium", device=device)


## 3. Discover audio files and transcribe

Auto-discover all `.wav` and `.mp3` files in `data/`. For each file we keep the
full transcript plus a per-utterance confidence proxy:
**`avg_logprob` averaged across all segments** (Whisper reports one
`avg_logprob` per segment; we mean-pool to get a single utterance-level value).

`word_timestamps=True` gives word-level timing for downstream prosody alignment
in notebook 03.


In [ ]:
audio_files: list[Path] = sorted(
    [p for ext in ("*.wav", "*.mp3") for p in DATA_DIR.glob(ext)]
)

if not audio_files:
    raise FileNotFoundError(
        f"No .wav or .mp3 files found in {DATA_DIR}. "
        "Add 5–20 short English clips and re-run."
    )


def transcribe_file(path: Path) -> dict[str, Any]:
    result = model.transcribe(
        str(path),
        language="en",
        word_timestamps=True,
        verbose=False,
    )
    segments = result.get("segments", []) or []
    if segments:
        avg_logprob = sum(s["avg_logprob"] for s in segments) / len(segments)
        duration_s = segments[-1]["end"]
    else:
        avg_logprob = None
        duration_s = 0.0
    return {
        "file": path.name,
        "transcript": result["text"].strip(),
        "avg_logprob": avg_logprob,
        "n_segments": len(segments),
        "duration_s": duration_s,
    }


records: list[dict[str, Any]] = [transcribe_file(p) for p in audio_files]


## 4. Save results and sanity-check

Write the per-utterance table to `output/01_whisper_results.csv` and print the
first few rows so we can eyeball that transcripts look reasonable and
`avg_logprob` values are in the expected range (typically between **-1.0** and
**-0.1** for clean speech; lower means less confident).


In [ ]:
df = pd.DataFrame(records)
csv_path = OUTPUT_DIR / "01_whisper_results.csv"
df.to_csv(csv_path, index=False)

print(f"Saved {len(df)} rows to {csv_path.relative_to(REPO_ROOT)}")
df.head(10)
